In [1]:
# Regression lineaire, Random forest, XGBoost, LightGBM, CatBoost, ExtraTrees​, PLS Regressor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import hydroeval as hd
from sklearn.base import clone
from pipeline import pipeline
import joblib
#plt.style.use("dark_background")

In [2]:
model=joblib.load("modele_linear_regression.pkl")

data=pd.read_excel("Data//DonneePreModel.xlsx").dropna()
data.head()

,Date,Pluie,Debit,Debit-j+1,Debit-j+2,Debit-j+3,Debit-j+4,Debit-j+5,Cum_Pluie_15,Cum_Pluie_30,...,Debit-j-2,Debit-j-3,Debit-j-4,Debit-j-5,Cum_Pluie_110,Cum_Pluie_130,Cum_Pluie_150,Cum_Pluie_170,Cum_Pluie_180,Cum_Pluie_200
5,1992-01-06,0.0,16.610001,16.430000,16.200001,16.129999,15.90,15.66,0.0,0.0,...,17.000000,17.150000,17.280001,17.280001,0.0,0.0,0.0,0.0,0.0,0.0
6,1992-01-07,0.0,16.430000,16.200001,16.129999,15.900000,15.66,15.61,0.0,0.0,...,16.889999,17.000000,17.150000,17.280001,0.0,0.0,0.0,0.0,0.0,0.0
7,1992-01-08,0.0,16.200001,16.129999,15.900000,15.660000,15.61,15.48,0.0,0.0,...,16.610001,16.889999,17.000000,17.150000,0.0,0.0,0.0,0.0,0.0,0.0
8,1992-01-09,0.0,16.129999,15.900000,15.660000,15.610000,15.48,15.33,0.0,0.0,...,16.430000,16.610001,16.889999,17.000000,0.0,0.0,0.0,0.0,0.0,0.0
9,1992-01-10,0.0,15.900000,15.660000,15.610000,15.480000,15.33,15.21,0.0,0.0,...,16.200001,16.430000,16.610001,16.889999,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
cible=["Debit-j+1","Debit-j+2","Debit-j+3"]
data["Annee"]=data.Date.dt.year
data["Mois"]=data.Date.dt.month
data["Jour"]=data.Date.dt.day
data=data.drop(["Debit-j+4","Debit-j+5"],axis=1)
train=data[data.Annee<=2014]
test=data[(2014<data.Annee) & (data.Annee<=2019)]
vald=data[(2019<data.Annee)]
X_vald=data[(2019<data.Annee)].drop(cible,axis=1)
y_vad=data[(2019<data.Annee)][cible]
X_train=train.drop(cible,axis=1)
X_test=test.drop(cible,axis=1)
y_train=train[cible]
y_test=test[cible]
X_train=X_train.drop("Annee",axis=1)
X_test=X_test.drop("Annee",axis=1)
X_va=X_vald.drop("Annee",axis=1)

In [4]:
data

,Date,Pluie,Debit,Debit-j+1,Debit-j+2,Debit-j+3,Cum_Pluie_15,Cum_Pluie_30,Cum_Pluie_45,Cum_Pluie_60,...,Debit-j-5,Cum_Pluie_110,Cum_Pluie_130,Cum_Pluie_150,Cum_Pluie_170,Cum_Pluie_180,Cum_Pluie_200,Annee,Mois,Jour
5,1992-01-06,0.0,16.610001,16.430000,16.200001,16.129999,0.000000,0.000000,0.000000,0.000000,...,17.280001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1992,1,6
6,1992-01-07,0.0,16.430000,16.200001,16.129999,15.900000,0.000000,0.000000,0.000000,0.000000,...,17.280001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1992,1,7
7,1992-01-08,0.0,16.200001,16.129999,15.900000,15.660000,0.000000,0.000000,0.000000,0.000000,...,17.150000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1992,1,8
8,1992-01-09,0.0,16.129999,15.900000,15.660000,15.610000,0.000000,0.000000,0.000000,0.000000,...,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1992,1,9
9,1992-01-10,0.0,15.900000,15.660000,15.610000,15.480000,0.000000,0.000000,0.000000,0.000000,...,16.889999,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1992,1,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11670,2023-12-22,0.0,27.854750,27.233536,26.551107,25.837714,8.382097,29.387790,72.351305,115.269493,...,31.483428,277.005812,277.005812,293.031215,364.814209,370.799414,497.245288,2023,12,22
11671,2023-12-23,0.0,27.233536,26.551107,25.837714,25.140571,8.382097,28.048255,71.592994,115.269493,...,30.841499,277.005812,277.005812,293.031215,361.216568,370.799414,497.245288,2023,12,23
11672,2023-12-24,0.0,26.551107,25.837714,25.140571,24.565679,8.382097,28.048255,60.340297,113.536826,...,30.226429,277.005812,277.005812,292.966684,359.036417,368.677514,496.814061,2023,12,24
11673,2023-12-25,0.0,25.837714,25.140571,24.565679,24.022215,8.222802,25.424902,48.727751,96.061740,...,29.512286,268.869420,277.005812,292.966684,358.888163,368.677514,486.537134,2023,12,25


In [5]:
x=pipeline(data[data.Annee<=2014],date_jour="2015-01-01",pluie_jour=0,debit_jour=8.824000).columns
x

Index(['Pluie', 'Debit', 'Cum_Pluie_15', 'Cum_Pluie_30', 'Cum_Pluie_45',
       'Cum_Pluie_60', 'Cum_Pluie_75', 'Cum_Pluie_90', 'Debit-j-1',
       'Debit-j-2', 'Debit-j-3', 'Debit-j-4', 'Debit-j-5', 'Cum_Pluie_110',
       'Cum_Pluie_130', 'Cum_Pluie_150', 'Cum_Pluie_170', 'Cum_Pluie_180',
       'Cum_Pluie_200', 'Mois', 'Jour'],
      dtype='str')

In [6]:
from pipeline import pipeline_predict

In [10]:
data[data.Date==pd.to_datetime("2021-03-01")]

,Date,Pluie,Debit,Debit-j+1,Debit-j+2,Debit-j+3,Cum_Pluie_15,Cum_Pluie_30,Cum_Pluie_45,Cum_Pluie_60,...,Debit-j-5,Cum_Pluie_110,Cum_Pluie_130,Cum_Pluie_150,Cum_Pluie_170,Cum_Pluie_180,Cum_Pluie_200,Annee,Mois,Jour
10644,2021-03-01,0.0,3.962,4.272,6.348,8.37,110.580649,114.37559,114.37559,121.367135,...,3.73,234.655025,359.357185,480.19149,705.516682,707.315052,801.053173,2021,3,1


In [12]:
pipeline_predict(data,date_jour="2021-03-01",pluie_jour=0,debit_jour=3.962,model=model)

array([[3.89185852, 3.50219529, 3.05556054]])